# Clustering : segmentation des clients (version standardisée)

Cas : KPMG_Customer_Demography

Ce notebook reprend la préparation du [notebook 01](01_segmentation_clients_kmeans.ipynb) (mêmes variables, même encodage, même K-Means) avec une seule différence : les variables numériques sont **standardisées** avant le K-Means.

Dans le notebook 01, les variables qui ont les plus grandes valeurs (`Wealth_score`, `Annual_purchase_frequency`) pèsent le plus dans le calcul des distances. La standardisation donne le même poids à chaque variable numérique.

## 1. Importation des données

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# Le fichier CSV se trouve dans le dossier data/ du dépôt.
# Sur Google Colab, remplacer par le chemin du fichier sur Drive.
DATA_PATH = Path('../data/KPMG_Customer_Demography.csv')

df = pd.read_csv(DATA_PATH)
df.shape

## 2. Nettoyage et création des variables

Même nettoyage et mêmes variables que dans le notebook 01 (l'exploration des données y est détaillée).

In [ ]:
df = df.drop(columns=['customer_id', 'First_Name', 'Last_Name', 'DOB', 'Postcode', 'Country', 'deceased_indicator', 'Address'], errors='ignore')

In [ ]:
df['Age_group'] = pd.cut(df['AGE'],
                         bins=[0, 34, 54, 120],
                         labels=['jeune', 'adulte', 'senior'])
df['Purchase_intensity'] = (df['past_3_years_bike_related_purchases'] / df['tenure']).round(2)
df['Wealth_score'] = df['past_3_years_bike_related_purchases'] * df['property_valuation']
df['Annual_purchase_frequency'] = df['past_3_years_bike_related_purchases'] * 3
df['Buyer_potential'] = pd.cut(df['tenure'],
                               bins=[0, 3, 7, 20],
                               labels=['Occasionnel', 'Regulier', 'Passionné'])
df.describe().round(2)

## 3. Standardisation

- Les variables catégorielles sont encodées en colonnes 0/1 (*one-hot*), comme dans le notebook 01.
- Seules les variables **numériques** sont standardisées (moyenne 0, écart-type 1). Les colonnes 0/1 restent telles quelles : les standardiser aussi donnerait un poids très fort aux modalités rares.

In [ ]:
df_encoded = pd.get_dummies(df, dtype=int)

colonnes_numeriques = df.select_dtypes(include='number').columns.tolist()
print('Variables standardisées :', colonnes_numeriques)

X_scaled = df_encoded.copy()
X_scaled[colonnes_numeriques] = StandardScaler().fit_transform(df_encoded[colonnes_numeriques])

X_scaled[colonnes_numeriques].describe().loc[['mean', 'std']].round(2)

## 4. Choix du nombre de clusters

In [ ]:
K = range(2, 10)
inerties, silhouettes = [], []
for k in K:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    etiquettes = km.fit_predict(X_scaled)
    inerties.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, etiquettes))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(K, inerties, 'bx-')
axes[0].set_title('Méthode du coude')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertie')
axes[1].plot(K, silhouettes, 'go-')
axes[1].set_title('Indice de silhouette')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette moyenne')
plt.tight_layout()
plt.show()

pd.DataFrame({'k': list(K), 'inertie': inerties, 'silhouette': silhouettes}).round(3)

Quel que soit k, l'indice de silhouette reste faible (inférieur à 0,2) : une fois les variables standardisées, les groupes sont peu séparés. **k = 3** est conservé pour pouvoir comparer avec le notebook 01 (la silhouette est presque identique à celle de k = 2).

In [ ]:
K_FINAL = 3
kmeans = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=10, random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X_scaled)

# Réduction de dimensionnalité avec PCA pour visualisation
pca = PCA(n_components=2, random_state=RANDOM_STATE)
data_2d = pca.fit_transform(X_scaled)
centres_2d = pca.transform(pd.DataFrame(kmeans.cluster_centers_, columns=X_scaled.columns))

plt.figure(figsize=(8, 5))
plt.scatter(data_2d[:, 0], data_2d[:, 1], c=labels, cmap='viridis', s=12, alpha=0.6)
plt.scatter(centres_2d[:, 0], centres_2d[:, 1], color='red', marker='x', s=100)
plt.title('Clusters K-Means (variables standardisées) avec PCA')
plt.xlabel(f'Composante principale 1 ({pca.explained_variance_ratio_[0]:.0%} de la variance)')
plt.ylabel(f'Composante principale 2 ({pca.explained_variance_ratio_[1]:.0%} de la variance)')
plt.show()

## 5. Profil des clusters

Les profils sont calculés sur les valeurs d'origine (non standardisées) pour rester lisibles.

In [ ]:
df_clustered = df.copy()
df_clustered['cluster_label'] = labels

variables = ['Wealth_score', 'Annual_purchase_frequency', 'property_valuation', 'AGE', 'tenure']
profil = df_clustered.groupby('cluster_label')[variables].mean().round(1)
profil.insert(0, 'effectif', df_clustered['cluster_label'].value_counts().sort_index())
profil.insert(1, 'part_%', (profil['effectif'] / len(df_clustered) * 100).round(1))
profil

In [ ]:
for col in ['wealth_segment', 'gender', 'Age_group', 'Buyer_potential']:
    print(f'Répartition de {col} par cluster (%)')
    display((pd.crosstab(df_clustered['cluster_label'], df_clustered[col], normalize='index') * 100).round(1))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, variables):
    sns.boxplot(x='cluster_label', y=col, data=df_clustered, ax=ax)
    ax.set_title(f'{col} par cluster')
    ax.set_xlabel('Cluster')
for ax in axes.flat[len(variables):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

## 6. Comparaison avec le notebook 01 (variables non standardisées)

In [ ]:
# K-Means du notebook 01 : mêmes variables encodées, sans standardisation
labels_brut = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=10, random_state=RANDOM_STATE).fit_predict(df_encoded)

ari = adjusted_rand_score(labels_brut, labels)
print(f'Indice de Rand ajusté entre les deux segmentations : {ari:.3f} (1 = identiques, 0 = sans lien)')

pd.crosstab(pd.Series(labels_brut, name='sans standardisation'),
            pd.Series(labels, name='avec standardisation'))

## 7. Analyse des résultats

Une fois les variables numériques standardisées, le K-Means (k = 3) ne classe plus les clients selon leur seul niveau d'achat : l'âge et l'ancienneté deviennent déterminants.

| Cluster | Profil | Clients | Part | Âge moyen | Ancienneté moyenne | `Wealth_score` (moyenne) | Fréquence d'achat (moyenne) |
|:-------:|--------|--------:|-----:|----------:|-------------------:|-------------------------:|----------------------------:|
| 0 | Jeunes clients récents | 586 | 15,0 % | 27,5 ans | 3,1 ans | 458 | 188 |
| 1 | Clients établis, achats faibles | 1 856 | 47,5 % | 47,0 ans | 11,5 ans | 165 | 74 |
| 2 | Clients établis, achats élevés | 1 466 | 37,5 % | 49,2 ans | 12,6 ans | 593 | 223 |

- **Cluster 0** : 89 % de moins de 35 ans, et 66 % de clients « occasionnels » selon l'ancienneté. Leur niveau d'achat est intermédiaire.
- **Clusters 1 et 2** : clients plus âgés et plus anciens, qui se distinguent surtout par leur volume d'achat.
- Le segment de richesse (`wealth_segment`) et le genre se répartissent presque de la même façon dans les trois clusters : ils n'expliquent pas la segmentation.

**Comparaison avec le notebook 01** : l'indice de Rand ajusté vaut 0,35, donc les deux segmentations sont en partie différentes. Environ 86 % des clients « faible valeur » du notebook 01 (1 474 sur 1 717) se retrouvent dans le cluster 1, et environ 80 % des clients « premium » (682 sur 854) dans le cluster 2. Les clients plus jeunes et plus récents, noyés dans les autres groupes sans standardisation, forment ici un segment à part.

**À interpréter avec prudence** :

- L'indice de silhouette est faible (0,179 pour k = 3) et les nuages de points de l'ACP se chevauchent : les clusters sont des profils moyens, pas des catégories nettes.
- `Annual_purchase_frequency` est un multiple exact des achats sur 3 ans, et `Wealth_score` en dépend aussi. Après standardisation, ces colonnes se répètent et comptent plusieurs fois dans les distances. Piste suivante : n'en garder qu'une ou deux et relancer le K-Means.

**Pistes d'action marketing** (à valider avec l'équipe) :

- **Cluster 0** : offres d'accueil et parcours d'intégration pour installer l'habitude d'achat chez les jeunes clients récents.
- **Cluster 1** : offres ciblées et remises liées à une carte de fidélité pour relancer les clients établis qui achètent peu.
- **Cluster 2** : service premium ou VIP pour fidéliser les clients établis qui achètent le plus.